# Lab 34: The Category Snap

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-34.ipynb)

**What you will do:** find where a synthetic ba-to-pa continuum switches category for you, repeat the judgement several times to see how consistent your own boundary is, compare it with the roughly 30 ms voice onset time boundary the book's figure reports, and train a simple vector quantiser to see a machine analogue of the same sharp, learned boundary.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 34 you listened to a continuum of synthetic syllables that vary in a single continuous
acoustic dimension, voice onset time (VOT) — the delay between a stop consonant's release and
the start of voicing — running from a short-VOT "ba" to a long-VOT "pa." Rather than hearing a
gradual blend, you heard a sharp switch from one category to the other at a boundary, even
though the underlying acoustic steps were equal-sized. Figure 4.2 places that idealised
boundary at about 30 ms.

## Record your results

Using the same or a similar VOT-continuum demo, repeat the identification judgement several
times: work through the continuum and note the step (or the VOT in milliseconds, if the demo
reports it) at which your percept switches from "ba" to "pa." Repeating it checks how
consistent your own boundary is from one pass to the next.

In [ ]:
# example data: replace with your own repeated boundary judgements, in milliseconds of VOT
my_boundary_ms = [28, 32, 25, 30, 29]  # example data: one value per repeated pass through the continuum

boundary = pd.Series(my_boundary_ms, name="perceived boundary (ms VOT)")
display(boundary.describe().to_frame())
boundary.plot(marker="o", linestyle="-", figsize=(6, 3))
plt.ylabel("perceived ba/pa boundary (ms)"); plt.xlabel("pass number"); plt.show()

## Compare

Figure 4.2 places the idealised identification boundary at about 30 ms of VOT. Compare your
own mean boundary with that figure — bearing in mind this is only a fair comparison if your
demo's continuum also spans roughly 0-60 ms in equal steps, as the book's figure does.

In [ ]:
mean_boundary = boundary.mean()
print(f"Your mean perceived boundary: {mean_boundary:.0f} ms (SD {boundary.std():.1f} ms across passes)")
if 20 <= mean_boundary <= 40:
    print("Close to the idealised ~30 ms boundary in Figure 4.2.")
else:
    print("Some distance from the idealised ~30 ms boundary -- category boundaries do shift"
          " with speech rate and phonetic context, and different demos use different voices"
          " and continua, so an exact match is not expected.")

## The AI side

The book's AI Connection compares categorical perception with **vector quantisation**: an
algorithm that learns a small set of discrete "codewords" for a continuous space (van den
Oord, Vinyals, & Kavukcuoglu, 2017), the same idea behind neural audio codecs. Below, a toy
one-dimensional vector quantiser is trained with scikit-learn's k-means on values spread
along a "VOT-like" axis, then asked to classify new, closely-spaced values. Check whether it
produces the same signature as your own hearing: a sharp switch between codewords, with a
boundary near the midpoint between two learned prototypes.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q scikit-learn

from sklearn.cluster import KMeans

# Training values cluster into two groups, "short VOT" and "long VOT", the way real speech does
rng = np.random.default_rng(0)
short_vot = rng.normal(10, 4, 60)
long_vot = rng.normal(50, 4, 60)
training_values = np.concatenate([short_vot, long_vot]).reshape(-1, 1)

quantiser = KMeans(n_clusters=2, n_init=10, random_state=0).fit(training_values)
codewords = sorted(quantiser.cluster_centers_.ravel())
boundary_ms = np.mean(codewords)
print(f"Learned codewords (ms VOT): {codewords[0]:.1f}, {codewords[1]:.1f}")
print(f"Implied boundary: {boundary_ms:.1f} ms")

test_vot = np.arange(0, 61, 2)
assigned = quantiser.predict(test_vot.reshape(-1, 1))
identification = pd.DataFrame({"VOT (ms)": test_vot, "assigned codeword": assigned})
display(identification)

plt.step(test_vot, assigned, where="mid", color="firebrick")
plt.axvline(boundary_ms, color="grey", linestyle="--", label=f"boundary ~{boundary_ms:.0f} ms")
plt.xlabel("VOT (ms)"); plt.ylabel("assigned codeword (0 or 1)"); plt.legend(); plt.show()

The quantiser's assignment should snap sharply from codeword 0 to codeword 1 right at the
midpoint between the two learned prototypes -- an abrupt step, not a gradual ramp, echoing
the sharp identification function in Figure 4.2. The parallel has real limits. The
quantiser's two clusters exist only because the training data happened to form two groups; a
human phoneme boundary is shaped by the phonology of a specific language (which is why
Japanese and English listeners draw different boundaries for other continua) and stays
useful even for sounds the listener has never heard before. The quantiser also has no
within-category graded sensitivity at all, whereas human within-category discrimination is
poor but rarely at chance.

## Pool the class data

Pool everyone's mean perceived boundary (anonymous IDs, never names) to see how much people
vary, and whether the class average sits near the book's idealised 30 ms figure.

In [ ]:
import io
csv_text = """id,mean_boundary_ms
P01,29
P02,34
P03,26
P04,31
P05,38
P06,24
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))

boot = [df.mean_boundary_ms.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean boundary: {df.mean_boundary_ms.mean():.0f} ms "
      f"(95% bootstrap interval {lo:.0f}-{hi:.0f} ms, n = {len(df)})")
df.mean_boundary_ms.plot.hist(bins=6, color="firebrick", alpha=0.7, figsize=(6, 3))
plt.axvline(30, color="grey", linestyle="--", label="Figure 4.2's idealised boundary")
plt.xlabel("perceived boundary (ms VOT)"); plt.legend(); plt.show()

## Questions to think about

1. The book notes that Japanese speakers, whose language does not distinguish /r/ and /l/, show a weaker category boundary on the English /r/-/l/ continuum than English speakers do. If you retrained the k-means quantiser above on training data drawn from only one broad, overlapping cluster instead of two separate ones, what would you expect the "boundary" to look like, and how is that similar to what happens for a contrast a listener's language does not use?
2. Real categorical perception is "strongly categorical... not perfectly categorical" — within-category discrimination is poor but rarely at chance. Does the k-means quantiser above have any equivalent of graded, sub-phonemic sensitivity within a category, or is its assignment purely binary?
3. Your own repeated passes through the continuum in the record step may not have landed on exactly the same boundary each time. What does this variability suggest about calling the boundary "sharp," given that the book itself says the identification function is steep but not infinitely steep?
4. The book states that the boundary's location itself shifts with speech rate and surrounding phonetic context. What would need to change in the k-means simulation above to model a boundary that moves depending on context, rather than staying fixed once trained?

## Challenge

If you can find a demo for a second continuum -- for example an English /r/-/l/ continuum, or
a vowel continuum -- try it yourself and compare how sharp the boundary feels compared with the
ba/pa boundary you measured here.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-34.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")